# Composite DCA State Score - Validation Notebook

**Phase 2 Diagnostic Analysis**

This notebook validates the composite score engine by analyzing:
1. Composite score vs forward 5d/10d returns
2. Regime-conditioned behavior heatmaps
3. Visual score decomposition
4. Score distribution by asset class

**Important**: This is a diagnostic tool, NOT for predictive PnL optimization.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
# Import custom modules
from data import DataFetcher
from score_engine import CompositeScorer, ScorerConfig
from backtester import DiagnosticBacktester, BacktestConfig
from indicators import IndicatorConfig, IndicatorEngine

# Initialize components
fetcher = DataFetcher(cache_dir='../data/cache')

scorer_config = ScorerConfig(
    r_thresh=0.5,
    S_scale=1.5,
    g_pers_k=10.0
)

backtest_config = BacktestConfig(
    forward_windows=[5, 10, 20, 60],
    n_quantiles=10,
    dca_high_threshold=70,
    dca_low_threshold=30
)

print("Modules loaded successfully!")

## 1. Data Fetching

Fetch historical data for our target assets:
- **Commodities**: XAU (Gold), XAG (Silver)
- **US Equities**: AAPL, NFLX
- **Indian Equities**: RELIANCE.NS, TCS.NS

In [ ]:
# Define assets
ASSETS = {
    'commodities': ['XAU', 'XAG'],
    'us_equities': ['AAPL', 'NFLX'],
    'indian_equities': ['RELIANCE.NS', 'TCS.NS']
}

ALL_SYMBOLS = sum(ASSETS.values(), [])

# Fetch data
data_dict = {}
meta_dict = {}

for symbol in ALL_SYMBOLS:
    print(f"Fetching {symbol}...")
    df, meta = fetcher.fetch_daily(symbol, period='max')
    if df is not None:
        data_dict[symbol] = df
        meta_dict[symbol] = meta
        print(f"  ✓ {len(df)} rows, {meta.get('source')}")
    else:
        print(f"  ✗ Failed to fetch")

print(f"\nFetched data for {len(data_dict)} assets")

## 2. Composite Score Computation

In [ ]:
# Compute scores for all assets
scores_dict = {}

for symbol, df in data_dict.items():
    print(f"Computing scores for {symbol}...")
    try:
        scorer = CompositeScorer(scorer_config)
        result = scorer.fit_transform(df, debug=True)
        scores_dict[symbol] = result
        
        summary = result.summary()
        print(f"  ✓ {summary['n_valid']} valid scores")
        print(f"    Mean: {summary['mean_score']:.1f}, Std: {summary['std_score']:.1f}")
    except Exception as e:
        print(f"  ✗ Error: {e}")

print(f"\nComputed scores for {len(scores_dict)} assets")

## 3. Score vs Forward Returns Analysis

In [ ]:
def plot_decile_returns(symbol, scores, prices, forward_days=10):
    """Plot average forward returns by score decile."""
    n = len(scores)
    
    # Compute forward returns
    fwd_returns = np.full(n, np.nan)
    for i in range(n - forward_days):
        if prices[i] > 0:
            fwd_returns[i] = (prices[i + forward_days] - prices[i]) / prices[i]
    
    # Filter valid
    valid_mask = ~np.isnan(scores) & ~np.isnan(fwd_returns)
    valid_scores = scores[valid_mask]
    valid_returns = fwd_returns[valid_mask]
    
    if len(valid_scores) < 100:
        print(f"Insufficient data for {symbol}")
        return None, None
    
    # Create deciles
    deciles = pd.qcut(valid_scores, 10, labels=False, duplicates='drop') + 1
    
    # Aggregate
    df = pd.DataFrame({'decile': deciles, 'return': valid_returns * 100})
    decile_means = df.groupby('decile')['return'].agg(['mean', 'std', 'count'])
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(decile_means)))
    bars = ax.bar(decile_means.index, decile_means['mean'], 
                  yerr=decile_means['std']/np.sqrt(decile_means['count']),
                  color=colors, edgecolor='black', alpha=0.8, capsize=3)
    
    ax.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel('Score Decile (1=Lowest, 10=Highest)', fontsize=12)
    ax.set_ylabel(f'Average {forward_days}d Forward Return (%)', fontsize=12)
    ax.set_title(f'{symbol}: Score Decile vs Forward Returns', fontsize=14, fontweight='bold')
    
    # Add correlation
    corr = np.corrcoef(decile_means.index, decile_means['mean'])[0, 1]
    ax.text(0.02, 0.98, f'Monotonicity: {corr:.2f}', transform=ax.transAxes, 
            fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    return fig, decile_means

# Plot for each asset
for symbol in list(scores_dict.keys())[:4]:
    result = scores_dict[symbol]
    df = data_dict[symbol]
    fig, decile_means = plot_decile_returns(symbol, result.scores, df['Close'].values)
    if fig:
        plt.show()
        print(f"\n{symbol} Decile Statistics:")
        print(decile_means.round(2))

In [ ]:
# Run full backtest on selected assets
backtester = DiagnosticBacktester(backtest_config)

for symbol in list(scores_dict.keys())[:2]:
    print(f"\n{'='*60}")
    print(f"Running backtest for {symbol}")
    print(f"{'='*60}")
    
    result = scores_dict[symbol]
    df = data_dict[symbol]
    
    bt_result = backtester.run_backtest(
        symbol=symbol,
        scores=result.scores,
        prices=df['Close'].values,
        dates=df.index
    )
    
    print(backtester.print_summary(bt_result))